In [2]:

from pathlib import Path
import urllib.request
import tarfile
import shutil

BASE_DIR = Path(r"D:\ConvNeXt_Project")
BASE_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE = BASE_DIR / "imagenette2-160.tgz"
DATA_DIR = BASE_DIR / "imagenette2-160"

URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"

# Hapus arsip hanya jika kosong
if ARCHIVE.exists() and ARCHIVE.stat().st_size == 0:
    ARCHIVE.unlink()
    print("Arsip kosong dihapus.")

# Unduh ulang jika arsip belum ada
if not ARCHIVE.exists():
    print("Mengunduh dataset ke drive D...")
    request = urllib.request.Request(
        URL,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    with urllib.request.urlopen(request, timeout=120) as response:
        with open(ARCHIVE, "wb") as output:
            shutil.copyfileobj(response, output)

# Validasi ukuran file
size_mb = ARCHIVE.stat().st_size / (1024 * 1024)
print(f"Ukuran arsip: {size_mb:.2f} MB")

if size_mb < 10:
    raise RuntimeError(
        "Arsip terlalu kecil. Unduhan mungkin gagal; periksa koneksi internet "
        "dan hapus arsip yang gagal sebelum mencoba kembali."
    )

# Periksa integritas arsip sebelum ekstraksi
if not tarfile.is_tarfile(ARCHIVE):
    raise RuntimeError(
        "File bukan arsip tar.gz yang valid. Hapus arsip yang gagal lalu unduh ulang."
    )

print("Mengekstrak dataset...")
with tarfile.open(ARCHIVE, "r:gz") as tar:
    tar.extractall(path=BASE_DIR, filter="data")

print("Selesai.")
print("Folder dataset:", DATA_DIR)
print("Dataset ditemukan:", DATA_DIR.exists())


Mengunduh dataset ke drive D...
Ukuran arsip: 94.42 MB
Mengekstrak dataset...
Selesai.
Folder dataset: D:\ConvNeXt_Project\imagenette2-160
Dataset ditemukan: True


In [3]:

from pathlib import Path

DATA_DIR = Path(r"D:\ConvNeXt_Project\imagenette2-160")

for split in ["train", "val"]:
    folder = DATA_DIR / split

    if folder.exists():
        images = [
            p for p in folder.rglob("*")
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        ]
        print(f"{split}: {len(images)} gambar")
    else:
        print(f"Folder {split} belum ditemukan.")


train: 9469 gambar
val: 3925 gambar


In [ ]:

%pip install torch torchvision

from pathlib import Path
from torchvision import datasets, transforms

# Lokasi dataset di drive D
DATA_DIR = Path(r"D:\ConvNeXt_Project\imagenette2-160")

# Pastikan folder dataset tersedia
assert (DATA_DIR / "train").exists(), "Folder train tidak ditemukan!"
assert (DATA_DIR / "val").exists(), "Folder val tidak ditemukan!"

# Transformasi untuk training
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(160),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Transformasi evaluasi: tanpa augmentasi acak
eval_transform = transforms.Compose([
    transforms.Resize(160),
    transforms.CenterCrop(160),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Dataset training
train_dataset = datasets.ImageFolder(
    DATA_DIR / "train",
    transform=train_transform
)

# Dataset evaluasi akhir
test_dataset = datasets.ImageFolder(
    DATA_DIR / "val",
    transform=eval_transform
)

print("Jumlah gambar train:", len(train_dataset))
print("Jumlah gambar val:", len(test_dataset))
print("Jumlah kelas:", len(train_dataset.classes))
print("Nama kelas:", train_dataset.classes)

assert len(train_dataset.classes) == 10, "Dataset seharusnya memiliki 10 kelas."
assert train_dataset.classes == test_dataset.classes, (
    "Urutan kelas train dan val tidak sama!"
)


In [ ]:

import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("GPU tersedia:", torch.cuda.is_available())
